In [6]:
import torch
from torch import nn
import gradio as gr
import pandas as pd
from sklearn.preprocessing import StandardScaler
import numpy as np


In [7]:
# Load dataset to rebuild scaler
df = pd.read_csv("StudentPerformanceFactors.csv")
df = pd.get_dummies(df, drop_first=True)

target = "Exam_Score"
X = df.drop(columns=[target]).values

scaler = StandardScaler()
scaler.fit(X)

# Recreate original model
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 1),
        )

    def forward(self, x):
        return self.model(x)

input_dim = X.shape[1]
model = NeuralNetwork(input_dim)
model.load_state_dict(torch.load("student_linear_regression.pth"))
model.eval()

NeuralNetwork(
  (model): Sequential(
    (0): Linear(in_features=27, out_features=1, bias=True)
  )
)

In [8]:
print(df.columns.tolist())
print(len(df.columns))

['Hours_Studied', 'Attendance', 'Sleep_Hours', 'Previous_Scores', 'Tutoring_Sessions', 'Physical_Activity', 'Exam_Score', 'Parental_Involvement_Low', 'Parental_Involvement_Medium', 'Access_to_Resources_Low', 'Access_to_Resources_Medium', 'Extracurricular_Activities_Yes', 'Motivation_Level_Low', 'Motivation_Level_Medium', 'Internet_Access_Yes', 'Family_Income_Low', 'Family_Income_Medium', 'Teacher_Quality_Low', 'Teacher_Quality_Medium', 'School_Type_Public', 'Peer_Influence_Neutral', 'Peer_Influence_Positive', 'Learning_Disabilities_Yes', 'Parental_Education_Level_High School', 'Parental_Education_Level_Postgraduate', 'Distance_from_Home_Moderate', 'Distance_from_Home_Near', 'Gender_Male']
28


In [9]:
# Prediction function
def predict(
    hours_studied,
    attendance,
    sleep_hours,
    previous_scores,
    tutoring_sessions,
    physical_activity,
    parental_involvement,
    access_to_resources,
    extracurricular,
    motivation,
    internet_access,
    family_income,
    teacher_quality,
    school_type,
    peer_influence,
    learning_disability,
    parental_education,
    distance_from_home,
    gender
):

    # Build feature list in correct order
    features = []

    # Numeric features
    features.append(hours_studied)
    features.append(attendance)
    features.append(sleep_hours)
    features.append(previous_scores)
    features.append(tutoring_sessions)
    features.append(physical_activity)

    # Parental involvement
    features += [
        1 if parental_involvement == "Low" else 0,
        1 if parental_involvement == "Medium" else 0,
    ]

    # Access to resources
    features += [
        1 if access_to_resources == "Low" else 0,
        1 if access_to_resources == "Medium" else 0,
    ]

    # Extracurricular
    features.append(1 if extracurricular == "Yes" else 0)

    # Motivation
    features += [
        1 if motivation == "Low" else 0,
        1 if motivation == "Medium" else 0,
    ]

    # Internet access
    features.append(1 if internet_access == "Yes" else 0)

    # Family income
    features += [
        1 if family_income == "Low" else 0,
        1 if family_income == "Medium" else 0,
    ]

    # Teacher quality
    features += [
        1 if teacher_quality == "Low" else 0,
        1 if teacher_quality == "Medium" else 0,
    ]

    # School type
    features.append(1 if school_type == "Public" else 0)

    # Peer influence
    features += [
        1 if peer_influence == "Neutral" else 0,
        1 if peer_influence == "Positive" else 0,
    ]

    # Learning disability
    features.append(1 if learning_disability == "Yes" else 0)

    # Parental education
    features += [
        1 if parental_education == "High School" else 0,
        1 if parental_education == "Postgraduate" else 0,
    ]

    # Distance from home
    features += [
        1 if distance_from_home == "Moderate" else 0,
        1 if distance_from_home == "Near" else 0,
    ]

    # Gender
    features.append(1 if gender == "Male" else 0)

    # Convert to array
    arr = np.array(features).reshape(1, -1)

    # Scale
    arr_scaled = scaler.transform(arr)

    # Convert to tensor
    tensor = torch.tensor(arr_scaled, dtype=torch.float32)

    # Predict
    with torch.no_grad():
        pred = model(tensor).item()

    return round(pred, 2)

In [11]:
# Gradio UI
examples = [
    [
        20, 90, 8, 85, 3, 5,
        "High", "High", "Yes", "High",
        "Yes", "High", "High", "Private",
        "Positive", "No", "Postgraduate",
        "Near", "Female"
    ],

    [
        8, 70, 6, 60, 0, 2,
        "Low", "Low", "No", "Low",
        "No", "Low", "Low", "Public",
        "Negative", "No", "High School",
        "Far", "Male"
    ]
]

demo = gr.Interface(
    fn=predict,

    inputs=[
        gr.Number(label="Hours Studied (hours per week)"),

        gr.Number(label="Attendance (%)"),

        gr.Number(label="Sleep Hours (hours per night)"),

        gr.Number(label="Previous Scores (0-100)"),

        gr.Number(label="Tutoring Sessions (per month)"),

        gr.Number(label="Physical Activity (hours per week)"),

        gr.Dropdown(
            ["Low", "Medium", "High"],
            label="Parental Involvement"
        ),

        gr.Dropdown(
            ["Low", "Medium", "High"],
            label="Access to Resources"
        ),

        gr.Dropdown(
            ["Yes", "No"],
            label="Participates in Extracurricular Activities"
        ),

        gr.Dropdown(
            ["Low", "Medium", "High"],
            label="Motivation Level"
        ),

        gr.Dropdown(
            ["Yes", "No"],
            label="Internet Access at Home"
        ),

        gr.Dropdown(
            ["Low", "Medium", "High"],
            label="Family Income"
        ),

        gr.Dropdown(
            ["Low", "Medium", "High"],
            label="Teacher Quality"
        ),

        gr.Dropdown(
            ["Public", "Private"],
            label="School Type"
        ),

        gr.Dropdown(
            ["Negative", "Neutral", "Positive"],
            label="Peer Influence"
        ),

        gr.Dropdown(
            ["Yes", "No"],
            label="Learning Disabilities"
        ),

        gr.Dropdown(
            ["High School", "College", "Postgraduate"],
            label="Parental Education Level"
        ),

        gr.Dropdown(
            ["Far", "Moderate", "Near"],
            label="Distance from Home"
        ),

        gr.Dropdown(
            ["Male", "Female"],
            label="Gender"
        ),
    ],

    outputs=gr.Textbox(
        label="Predicted Exam Score"
    ),

    title="Student Exam Score Predictor",

    description="""
This demo predicts a student's Exam_Score using a trained PyTorch linear regression model.
The prediction is based on the synthetic Student Performance Factors dataset used in this study.

The output represents the model's predicted Exam_Score and should be interpreted as a prediction from a model trained on synthetic data, not as educational advice.
""",

    examples=examples,
)

demo.launch()


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ba3cc5fae4e1b3933b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
